# Acquirium quick start

Acquirium is a metadata + timeseries platform for water systems. The server holds two things about a plant:

1. **a semantic model** — equipment, piping, sensors and units, described with the ASHRAE 223 / WaTr ontologies
2. **the timeseries** of every measured point

The key idea: **you don't query tag names, you query meaning.** You describe what you want in domain terms — a pump, salt concentration, whatever is upstream of the RO — and acquirium finds the points. Code written this way is not specific to a plant: it runs on any plant that has a model.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Setup
0. Follow the steps in [deployments/WATERTAP/README.md](../../deployments/WATERTAP/README.md) to install requirements
1. Start the server with a config: `acquirium server --config deployments/WATERTAP/scripts/acquirium.toml`
2. Connect:

In [ ]:
from acquirium import Acquirium
acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

########################################################################
########################################################################
########################################################################
## For better display of polars dataframes in Jupyter notebooks
import polars as pl
pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)

## Find entities by ontology classes
`entity()` takes plain text. The server resolves it against the ontology, so you don't need to know any ontology URIs to explore:

In [ ]:
q = acq.query().entity("Pump")
q.metadata()

## A query is a description, data comes last
Queries are built step by step and nothing is fetched until you ask. `.metadata()` tells you *which points* matched; `.dataframe()` then pulls the numbers. Let's ask for every salt concentration in the plant:

In [ ]:
q = (acq.query().measurement().where(substance = "constituent salt", quantity_kind = "mass_concentration"))
q.metadata()

Three points matched: 
- seawater feed
- brine discharge 
- product water 

We can access to their timeseries values with:

In [ ]:
q.dataframe(shape="wide", cast_value="float").tail(3)

## The topology is queryable
For questions about *where* such as what feeds this unit, what is measured downstream of that one, etc.; We can explore and find the related equipment. For instance, let's find the RO membrane and its upstream pump:

In [ ]:
q = acq.query().entity("reverse osmosis membrane").related("pump",direction="upstream")
q.metadata()

## Units
Every property can carry a QUDT unit and Acquirium can convert these units to each other:

In [ ]:
q = (acq.query().measurement().where(substance = "constituent salt", quantity_kind = "mass_concentration"))
data = q.data()
print(data.units())
data.convert_to("mg/L").dataframe().tail(3)

## Where to go next
- `explore-demo.ipynb` — one cell per feature of the query interface
- `watertap-1.ipynb` — the client reference: every feature, plus the internals (text resolution, generated SPARQL)
- `regulation.ipynb` — a real application: checking this plant against the California Ocean Plan and Title 22 drinking water standards
- `soft-sensor.ipynb` — a linear soft sensor for membrane fouling